In [2]:
import pandas as pd
import json
#EXTRACCIÓN
# Cargamos los datos (el archivo se llama 'TelecomX_Data.json')
with open('TelecomX_Data.json', 'r') as f:
    data = json.load(f)

# Convertimos a DataFrame inicial
df_raw = pd.DataFrame(data)

# Visualizamos la estructura inicial
print(f"Columnas originales: {df_raw.columns.tolist()}")
df_raw.head()


Columnas originales: ['customerID', 'Churn', 'customer', 'phone', 'internet', 'account']


,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


TRANSFORMACIÓN, LIMPIEZA Y APLANADO

In [3]:
# 1. Aplanar los datos anidados
# Esto expande 'customer', 'phone', 'internet' y 'account' en columnas como 'customer.gender'
df = pd.json_normalize(data)

# 2. Limpieza de nombres de columnas (opcional pero recomendado)
# Quitamos los puntos de los nombres de las columnas para facilitar el manejo
df.columns = [col.replace('.', '_') if '.' in col else col for col in df.columns]

# 3. Tratamiento de tipos de datos
# 'Charges_Total' suele venir como string en JSON, lo convertimos a numérico
df['account_Charges_Total'] = pd.to_numeric(df['account_Charges_Total'], errors='coerce')

# 4. Manejo de valores nulos
# Si el Total es nulo tras la conversión (por espacios en blanco), lo llenamos con 0 o la media
df['account_Charges_Total'] = df['account_Charges_Total'].fillna(0)

# 5. Estandarización de variables categóricas
# Ejemplo: Convertir 'Churn' vacío a 'No' (si aplica)
df['Churn'] = df['Churn'].replace('', 'No')

# 6. Crear nuevas métricas (Feature Engineering)
# Ejemplo: Gasto promedio real por mes de vida del cliente
df['avg_expenditure_per_tenure'] = df['account_Charges_Total'] / df['customer_tenure']
df['avg_expenditure_per_tenure'] = df['avg_expenditure_per_tenure'].fillna(0)

print("Datos transformados exitosamente.")
df.info()

Datos transformados exitosamente.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerID                  7267 non-null   object 
 1   Churn                       7267 non-null   object 
 2   customer_gender             7267 non-null   object 
 3   customer_SeniorCitizen      7267 non-null   int64  
 4   customer_Partner            7267 non-null   object 
 5   customer_Dependents         7267 non-null   object 
 6   customer_tenure             7267 non-null   int64  
 7   phone_PhoneService          7267 non-null   object 
 8   phone_MultipleLines         7267 non-null   object 
 9   internet_InternetService    7267 non-null   object 
 10  internet_OnlineSecurity     7267 non-null   object 
 11  internet_OnlineBackup       7267 non-null   object 
 12  internet_DeviceProtection   7267 non-null   object 
 13 